# Lab 6.7 &mdash; RAG with Citations

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 2 &middot; Module 6 &mdash; Agentic RAG**

### What you'll do
- Put the source and page INTO the context, because a model cannot cite what it cannot see
- Prompt for citations in a fixed format, and check they point at real documents
- Build a category-scoped chain from a filtered retriever
- Catch the failure that matters: a citation that is right next to an answer it does not support

> **How this lab works.** You write real LangChain code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those check the *objects you built* (a
> `Document`, a Chroma collection, a retriever, a chain), so they are deterministic and do not
> depend on the chat model. Cells marked **Run it for real** put your code in front of the
> sandbox model; that is the part worth watching. The score line is feedback, not a grade.

> **Two different models are in play, and only one of them is billed.** The **chat model**
> (`qwen36-35b-a3b-lab`) answers questions and is reached over the gateway. The **embedding
> model** (`all-MiniLM-L6-v2`, 384 dimensions) turns text into vectors and runs on this pod's
> own CPU &mdash; no key, no gateway, no tokens. Keeping them straight is most of Module 6.

> **The citation is the deliverable.** An answer nobody can check is worth less than
> no answer, and this is the one lab in the module a compliance reviewer would ask for.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, math, textwrap, warnings
from typing import Any, Callable

warnings.filterwarnings("ignore")     # sentence-transformers is chatty on first import

WORK = os.path.join("/tmp", "awmas-lab-6-07")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the CHAT model: qwen, through the sandbox gateway -------------------
# Already configured -- nothing to install, no key to register. Read from the
# environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Thinking is off by default here because you will make a lot of calls today;
# pass think=True to any call below to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Chat model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

# ---- the EMBEDDING model: local, free, nothing to configure --------------
# all-MiniLM-L6-v2, 384 dimensions. It runs on this pod's CPU and has nothing to do with
# the chat model above: no gateway, no key, no tokens billed. The cache is already warm
# in your sandbox, so the first call is a second or two, not a download.
#
# It is reached through onnxruntime rather than torch, and that is a measured choice
# rather than a taste: same model, same vectors, ~170 MB of memory instead of ~840. Your
# whole sandbox has 2.5 GB for every notebook you leave open, and a kernel you have
# forgotten about is still holding its share.
from langchain_core.embeddings import Embeddings

class MiniLMEmbeddings(Embeddings):
    """all-MiniLM-L6-v2 behind LangChain's Embeddings interface.

    Two methods is the whole contract -- which is why a store, a splitter and a chain
    never need to know which model is underneath, or what runtime it uses."""

    def __init__(self):
        from chromadb.utils.embedding_functions import ONNXMiniLM_L6_V2
        self._fn = ONNXMiniLM_L6_V2()

    def embed_documents(self, texts: list) -> list:
        return [[float(x) for x in v] for v in self._fn(list(texts))]

    def embed_query(self, text: str) -> list:
        return [float(x) for x in self._fn([text])[0]]


_emb_cache = {}
def get_embeddings():
    """The embedding model, built once per kernel."""
    if "model" not in _emb_cache:
        _emb_cache["model"] = MiniLMEmbeddings()
    return _emb_cache["model"]

print("work dir   :", WORK)
print("chat model :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the corpus (synthetic, self-contained)
# Ten short passages from a company handbook. Note what each one carries besides its text:
# a category, a source file and a page. Those three are what Lab 6.3 filters on and what
# Lab 6.7 cites -- metadata is not decoration, it is the half of retrieval that is exact.
#
# Note also what is NOT here: nothing mentions salary, notice period or the share price.
# Labs 6.6 and 6.8 need that gap, because refusing is a feature.

HANDBOOK = [
    {"text": "Annual leave is 24 days per year for full-time employees. Leave must be applied "
             "for at least 3 working days in advance. Unused annual leave cannot be carried "
             "forward to the next financial year.",
     "category": "leave", "source": "handbook.pdf", "page": 5},
    {"text": "Sick leave is 12 days per year. Notify your manager by 10 AM on the day of "
             "absence. A medical certificate is required for absences of more than 2 "
             "consecutive days.",
     "category": "leave", "source": "handbook.pdf", "page": 5},
    {"text": "Maternity leave is 26 weeks of paid leave. Paternity leave is 2 weeks. Both "
             "must be applied for at least 30 days before the expected date.",
     "category": "leave", "source": "handbook.pdf", "page": 6},
    {"text": "Employees may work from home up to 3 days per week with team lead approval. "
             "Core hours are 10 AM to 4 PM IST, and you must be reachable during them.",
     "category": "wfh", "source": "handbook.pdf", "page": 8},
    {"text": "A VPN connection is mandatory for reaching internal systems from home. "
             "Contact the IT helpdesk for VPN setup.",
     "category": "wfh", "source": "handbook.pdf", "page": 8},
    {"text": "Internet reimbursement is 1,500 per month for employees working from home. "
             "Submit the broadband bill to finance by the 5th of each month.",
     "category": "expense", "source": "handbook.pdf", "page": 9},
    {"text": "Travel expenses must be submitted with original receipts within 7 working days "
             "of travel. The meal allowance during client visits is 500 per day.",
     "category": "expense", "source": "handbook.pdf", "page": 12},
    {"text": "Laptops are provided by the company and replaced every 3 years. Software "
             "licence requests go through the IT helpdesk and must not be bought directly.",
     "category": "tech", "source": "tech-guide.pdf", "page": 7},
    {"text": "The backend stack is Python with FastAPI, and Java with Spring Boot. New "
             "services should use Python unless there is a specific reason not to. "
             "PostgreSQL is the primary database.",
     "category": "tech", "source": "tech-guide.pdf", "page": 3},
    {"text": "The Bangalore office is the headquarters, on the 5th floor, with 200+ staff. "
             "The Mumbai office is in the Worli business district, Tower A, 12th floor.",
     "category": "office", "source": "office-directory.pdf", "page": 15},
]

print(f"{len(HANDBOOK)} passages, "
      f"{len({d['category'] for d in HANDBOOK})} categories, "
      f"{len({d['source'] for d in HANDBOOK})} source files")

In [ ]:
# ------------------------------------------------- the corpus as LangChain Documents
from langchain_core.documents import Document

def handbook_documents() -> list:
    """One Document per passage: the text, and everything else as metadata."""
    return [Document(page_content=d["text"],
                     metadata={"category": d["category"],
                               "source": d["source"],
                               "page": d["page"]})
            for d in HANDBOOK]

print(len(handbook_documents()), "Document objects")

In [ ]:
# ------------------------------------------------- carried forward from Labs 6.5-6.6 (given)
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

_store = {}
def store():
    if "s" not in _store:
        _store["s"] = Chroma.from_documents(documents=handbook_documents(),
                                            embedding=get_embeddings(),
                                            collection_name="handbook_cite")
    return _store["s"]

def retriever(k: int = 3, flt: dict | None = None):
    kwargs = {"k": k}
    if flt:
        kwargs["filter"] = flt
    return store().as_retriever(search_kwargs=kwargs)

print("store and retriever ready")

## Concept

A citation is not a footnote you add afterwards. It is a **binding**: the model can only
name a source that was in the context it was given, so the source has to be *in* the
context.

That is two changes to Lab 6.6, both small:

1. `format_docs` writes `[handbook.pdf, page 5]` above each chunk instead of just joining text.
2. The prompt asks for the citation in a fixed shape.

And one thing that does not follow: a citation being *present* does not make the sentence
next to it *true*. The last section of this lab is about telling those apart.

## Section 1 &mdash; Putting the source in the context

Every `Document` already carries `source` and `page` &mdash; you decided that back in
Lab 6.3, which is why it is available now.

In [ ]:
def label_for(doc) -> str:
    """The one-line header that goes above a chunk in the context.

    Return it in the form:  [handbook.pdf, page 5]
    Use doc.metadata; fall back to "?" for a missing page."""
    return f"[{doc.metadata.get('source', 'unknown')}, page {doc.metadata.get('page', '?')}]"


def format_docs_with_sources(docs) -> str:
    """Given -- your label, then the text, blank line between chunks."""
    return "\n\n".join(f"{label_for(d)}\n{d.page_content}" for d in docs)

In [ ]:
# --- Self-check: Section 1   (string shapes -- no gateway)
def sample():
    return retriever().invoke("How many sick days do I get?")

check("the label names the source file",
      lambda: "handbook.pdf" in label_for(sample()[0]))
check("the label names the page",
      lambda: "page" in label_for(sample()[0]).lower()
              and any(ch.isdigit() for ch in label_for(sample()[0])))
check("a Document with no page does not crash the label",
      lambda: isinstance(label_for(Document(page_content="x", metadata={"source": "a.pdf"})), str),
      "real corpora have gaps -- a loader that skipped a page number must not break citing")
check("the formatted context carries every retrieved chunk's source",
      lambda: all(d.metadata["source"] in format_docs_with_sources(sample()) for d in sample()))
check("and still carries the text itself",
      lambda: all(d.page_content[:30] in format_docs_with_sources(sample()) for d in sample()))

guard(lambda: print(format_docs_with_sources(sample())[:400], "..."))

## Section 2 &mdash; Asking for the citation, and scoping the search

Two decisions left: how to ask for the citation, and what a scoped chain looks like when
you want an assistant that only ever answers from one part of the corpus.

In [ ]:
def citation_instruction() -> str:
    """Tell the model to cite. Name the FORMAT you want -- "cite your sources" gets you
    a different shape every call, which nothing downstream can parse."""
    return ("After each fact, cite the source it came from in parentheses, "
            "like (handbook.pdf, p.5).")


def leave_filter() -> dict:
    """The filter for a leave-only assistant."""
    return {"category": "leave"}


def citation_prompt():
    """Given."""
    return ChatPromptTemplate.from_template(
        "Answer using ONLY the context below. " + citation_instruction() +
        " If the context does not contain the answer, say you do not have that information."
        "\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:")


def chain_over(rtv):
    """Given -- any retriever, same chain."""
    return ({"context": rtv | format_docs_with_sources, "question": RunnablePassthrough()}
            | citation_prompt() | get_llm() | StrOutputParser())

In [ ]:
# --- Self-check: Section 2   (prompt and retriever objects)
check("the instruction names a citation format, not just the idea",
      lambda: "(" in citation_instruction() and ")" in citation_instruction(),
      "show the model the shape you want -- an example is worth a paragraph of description")
check("it mentions a real file from the corpus",
      lambda: any(s in citation_instruction()
                  for s in {d["source"] for d in HANDBOOK}))
check("the prompt still takes context and question",
      lambda: set(citation_prompt().input_variables) == {"context", "question"})
check("the leave-scoped retriever returns only leave passages",
      lambda: {d.metadata["category"]
               for d in retriever(k=3, flt=leave_filter()).invoke("What are my options?")}
              == {"leave"})
check("it finds all three leave passages, not just the one about annual leave",
      lambda: len(retriever(k=3, flt=leave_filter()).invoke("types of leave")) == 3)

In [ ]:
# --- Run it for real -------------------------------------------------------
def cited_answers():
    chain = chain_over(retriever())
    for q in ["How many sick days do I get, and do I need a doctor's note?",
              "How do I claim travel expenses?"]:
        print(f"Q: {q}\nA: {chain.invoke(q).strip()}\n")

if llm_ready():
    guard(cited_answers)

In [ ]:
# --- Run it for real: a scoped assistant -----------------------------------
def leave_desk():
    scoped = chain_over(retriever(k=3, flt=leave_filter()))
    q = "What kinds of leave can I take?"
    print(f"[leave-only] Q: {q}\nA: {scoped.invoke(q).strip()}\n")
    # ...and the same assistant asked something outside its scope. The retriever still
    # returns three leave passages, because a filter narrows -- it never returns nothing.
    q2 = "How much is the internet reimbursement?"
    print(f"[leave-only] Q: {q2}\nA: {scoped.invoke(q2).strip()}")
    print("\n  what it retrieved for that second question:")
    for d in retriever(k=3, flt=leave_filter()).invoke(q2):
        print(f"    ({d.metadata['category']}) {d.page_content[:56]}...")

if llm_ready():
    guard(leave_desk)

In [ ]:
# --- Run it for real: is the citation real? --------------------------------
# A citation is checkable, so check it. Every source the answer names must be one
# that was actually in the context -- a model that invents "policy-2019.pdf" is
# doing the most dangerous thing in this module.
def citations_are_real():
    q = "How many sick days do I get?"
    docs = retriever().invoke(q)
    answer = chain_over(retriever()).invoke(q)
    given = {d.metadata["source"] for d in docs}
    named = {s for s in {d["source"] for d in HANDBOOK} if s in answer}
    print("  in the context:", sorted(given))
    print("  named in the answer:", sorted(named) or "(none)")
    print("  every cited source was really there:", named <= given)

if llm_ready():
    guard(citations_are_real)

In [ ]:
score()

## Your turn

1. Add the category to `label_for` and ask the model to cite it too. Longer citations are
   not obviously better &mdash; at what point does the citation crowd out the answer?
2. Write the check from the last cell as a function that returns `True`/`False` and run it
   over ten questions. You have just built the first metric of Module 7, and it is the one
   that catches a fabricated source.
3. Harder: ask &ldquo;how many sick days, and can I carry them forward?&rdquo;. The handbook
   answers the first half and not the second. Read the answer very carefully &mdash; a
   correct citation sitting beside an unsupported clause is the failure mode that survives
   every demo, because everything on the screen looks right.